# 02 - Clustering de Patrones de Conducción

## Pregunta de negocio: ¿Existen estilos de conducción distintos?

---

### Diferencia con la segmentación de clientes

En el notebook anterior ([01 - Segmentación de Clientes](01_customer_segmentation.ipynb))
combinamos variables de **comportamiento de conducción + demográficas** para encontrar
segmentos de clientes con fines de marketing y negocio.

Aquí nos enfocamos **exclusivamente en la telemetría** para descubrir patrones de conducción
puros, sin influencia demográfica. Esto nos permite:

- Identificar estilos de manejo independientes del perfil del comprador.
- Comparar los patrones descubiertos con los estilos **auto-reportados** en encuestas.
- Diseñar programas de asistencia al conductor basados en comportamiento real.

### Técnicas utilizadas

- **K-Means** con método del codo y coeficiente de silueta.
- **PCA** (lineal) vs **t-SNE** (no lineal) para visualización:
  - PCA preserva la **varianza global** y las distancias a gran escala.
  - t-SNE preserva las **relaciones locales** (vecinos cercanos), revelando mejor la estructura de clusters.
- **Gráficos radar** para perfilar cada estilo de conducción.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import pi
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

vtype_colors = {
    'electrico': '#2ecc71',
    'gasolina': '#3498db',
    'hibrido': '#9b59b6',
    'deportivo': '#e74c3c'
}

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
print(f"Directorio raíz del proyecto: {project_root}")

---
## 2. Carga de datos de telemetría agregados

Cargamos las features de telemetría agregadas por vehículo. Las variables clave para
detectar patrones de conducción son:

| Feature | Descripción |
|---------|-------------|
| `speed_mean` | Velocidad media (km/h) |
| `speed_std` | Variabilidad de velocidad |
| `acceleration_std` | Variabilidad de aceleración |
| `harsh_braking_count` | Frenadas bruscas |
| `harsh_accel_count` | Aceleraciones bruscas |
| `consumption_mean` | Consumo medio |
| `night_ratio` | Proporción de conducción nocturna |
| `highway_ratio` | Proporción en autopista |

In [ ]:
# Cargar features de ML (agregadas por vehículo)
features_path = os.path.join(project_root, "data/processed/features_ml.csv")
df = pd.read_csv(features_path)
print(f"Dataset cargado: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
df.head()

In [ ]:
# Seleccionar features de conducción
driving_candidates = [
    'speed_mean', 'speed_std', 'acceleration_std',
    'harsh_braking_count', 'harsh_accel_count',
    'consumption_mean', 'night_ratio', 'highway_ratio',
    'avg_speed_kmh', 'std_speed_kmh', 'avg_acceleration_ms2',
    'std_acceleration_ms2', 'avg_consumption', 'max_speed_kmh',
    'harsh_braking_rate', 'harsh_accel_rate',
    'night_driving_ratio', 'highway_ratio_pct'
]

driving_features = [c for c in driving_candidates if c in df.columns]

# Si no hay features específicas, usar numéricas excluyendo IDs y coordenadas
if len(driving_features) < 3:
    exclude = ['vehicle_id', 'trip_id', 'gps_lat', 'gps_lon', 'age',
               'satisfaction_score', 'income_numeric', 'would_recommend']
    driving_features = [c for c in df.select_dtypes(include=[np.number]).columns
                        if c not in exclude]

print(f"Features de conducción seleccionadas ({len(driving_features)}):")
print(driving_features)

df_driving = df[driving_features].dropna()
print(f"\nVehículos para clustering: {len(df_driving)}")

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_driving)
print("Datos normalizados con StandardScaler.")

---
## 3. K-Means en features de conducción

Determinamos el número óptimo de clusters mediante el método del codo y silueta.

In [ ]:
k_range = range(2, 11)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    print(f"  k={k}: Inercia={km.inertia_:.1f}, Silueta={sil_scores[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Número de clusters (k)', fontsize=12)
axes[0].set_ylabel('Inercia', fontsize=12)
axes[0].set_title('Método del Codo', fontsize=14, fontweight='bold')
axes[0].set_xticks(list(k_range))

best_k = list(k_range)[np.argmax(sil_scores)]
colors_bar = ['#e74c3c' if k == best_k else '#3498db' for k in k_range]
axes[1].bar(k_range, sil_scores, color=colors_bar, edgecolor='white')
axes[1].set_xlabel('Número de clusters (k)', fontsize=12)
axes[1].set_ylabel('Coeficiente de Silueta', fontsize=12)
axes[1].set_title('Silueta por k', fontsize=14, fontweight='bold')
axes[1].set_xticks(list(k_range))

fig.suptitle('Selección de k para patrones de conducción', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

optimal_k = best_k
print(f"\nk óptimo según silueta: {optimal_k} (score={max(sil_scores):.4f})")

In [ ]:
# Ajustar K-Means final
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_driving = df_driving.copy()
df_driving['driving_cluster'] = kmeans.fit_predict(X_scaled)

print(f"K-Means ajustado con k={optimal_k}")
print(f"Silueta: {silhouette_score(X_scaled, df_driving['driving_cluster']):.4f}")
print(f"\nDistribución:")
print(df_driving['driving_cluster'].value_counts().sort_index())

---
## 4. Visualización de Clusters: PCA vs t-SNE

Comparamos dos técnicas de reducción de dimensionalidad:

- **PCA**: transformación lineal que maximiza la varianza. Bueno para ver la estructura global.
- **t-SNE**: transformación no lineal que preserva vecindades locales. Mejor para revelar
  clusters compactos y separados, pero no preserva distancias absolutas y puede crear
  artefactos visuales. El parámetro `perplexity` controla el balance entre estructura local/global.

In [ ]:
# PCA 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# t-SNE 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_scaled) - 1),
            n_iter=1000, learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X_scaled)

print(f"PCA varianza explicada: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, "
      f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"t-SNE KL divergence: {tsne.kl_divergence_:.4f}")

In [ ]:
# Visualización lado a lado
cluster_colors_arr = plt.cm.Set2(np.linspace(0, 1, optimal_k))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# PCA
for i in range(optimal_k):
    mask = df_driving['driving_cluster'] == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[cluster_colors_arr[i]], label=f'Cluster {i}',
                    alpha=0.6, s=50, edgecolors='white', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
axes[0].set_title('PCA 2D', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

# t-SNE
for i in range(optimal_k):
    mask = df_driving['driving_cluster'] == i
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=[cluster_colors_arr[i]], label=f'Cluster {i}',
                    alpha=0.6, s=50, edgecolors='white', linewidth=0.5)
axes[1].set_xlabel('t-SNE Dim 1', fontsize=12)
axes[1].set_ylabel('t-SNE Dim 2', fontsize=12)
axes[1].set_title('t-SNE 2D', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

fig.suptitle('Patrones de conducción: PCA vs t-SNE', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Perfiles de Estilos de Conducción

Analizamos cada cluster para darle un nombre descriptivo que refleje el patrón de conducción.

In [ ]:
# Perfil medio por cluster (valores originales)
profile = df_driving.groupby('driving_cluster')[driving_features].mean()
print("Perfil medio por cluster:")
display(profile.round(2))

In [ ]:
# Gráfico radar por cluster
def radar_chart(data, categories, title, ax, colors=None):
    """Crea un gráfico radar."""
    N = len(categories)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]

    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    ax.set_rlabel_position(0)
    plt.xticks(angles[:-1], categories, size=8)

    if colors is None:
        colors = plt.cm.Set2(np.linspace(0, 1, len(data)))

    for idx, (label, values) in enumerate(data.items()):
        vals = values.tolist()
        vals += vals[:1]
        ax.plot(angles, vals, 'o-', linewidth=2, label=label, color=colors[idx])
        ax.fill(angles, vals, alpha=0.1, color=colors[idx])

    ax.set_title(title, size=13, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# Normalizar a [0, 1]
profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-10)

# Top features más discriminantes
feat_std = profile_norm.std()
top_feats = feat_std.nlargest(min(8, len(driving_features))).index.tolist()

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))
radar_data = {f'Cluster {i}': profile_norm.loc[i, top_feats] for i in range(optimal_k)}
radar_chart(radar_data, top_feats, 'Perfiles de Estilo de Conducción', ax)
plt.tight_layout()
plt.show()

In [ ]:
# Asignar nombres descriptivos a cada patrón de conducción
pattern_names = [
    "Agresivo urbano",
    "Conservador de carretera",
    "Mixto eficiente",
    "Veloz nocturno",
    "Económico prudente",
    "Deportivo intenso",
    "Rutinario moderado"
]

print("=" * 60)
print("PATRONES DE CONDUCCIÓN DESCUBIERTOS")
print("=" * 60)

for i in range(optimal_k):
    name = pattern_names[i] if i < len(pattern_names) else f"Patrón {i}"
    cluster_profile = profile_norm.loc[i]
    top_high = cluster_profile.nlargest(3)
    top_low = cluster_profile.nsmallest(2)
    size = (df_driving['driving_cluster'] == i).sum()
    pct = size / len(df_driving) * 100

    print(f"\nCluster {i}: \"{name}\"")
    print(f"  Tamaño: {size} vehículos ({pct:.1f}%)")
    print(f"  Características dominantes: {', '.join(top_high.index)}")
    print(f"  Características bajas: {', '.join(top_low.index)}")

---
## 6. Cruce con datos de encuestas

¿Coinciden los patrones descubiertos por el algoritmo con los estilos de conducción
que los propios conductores reportaron en las encuestas? Esta comparación nos permite
evaluar la conciencia que tienen los conductores sobre su propio comportamiento.

In [ ]:
# Cargar datos de encuestas y combinar
survey_path = os.path.join(project_root, "data/raw/surveys/buyer_surveys.csv")
merged_path = os.path.join(project_root, "data/processed/vehicle_survey_merged.csv")

df_merged = None

if os.path.exists(merged_path):
    df_merged = pd.read_csv(merged_path)
    print(f"Cargado vehicle_survey_merged.csv: {df_merged.shape}")
elif os.path.exists(survey_path):
    df_survey = pd.read_csv(survey_path)
    print(f"Cargado buyer_surveys.csv: {df_survey.shape}")
    # Intentar merge por vehicle_id si existe
    if 'vehicle_id' in df.columns and 'vehicle_id' in df_survey.columns:
        df_merged = df.merge(df_survey, on='vehicle_id', how='left')
    else:
        # Alinear por índice
        df_merged = pd.concat([df.reset_index(drop=True),
                               df_survey.reset_index(drop=True)], axis=1)
    print(f"Merged: {df_merged.shape}")
else:
    print("No se encontraron datos de encuestas.")

if df_merged is not None:
    # Asignar clusters a los datos merged
    df_merged = df_merged.loc[df_driving.index[:len(df_merged)]].copy()
    df_merged['driving_cluster'] = df_driving['driving_cluster'].values[:len(df_merged)]

In [ ]:
# Crosstab: clusters descubiertos vs driving_style reportado
if df_merged is not None and 'driving_style' in df_merged.columns:
    ct = pd.crosstab(df_merged['driving_cluster'], df_merged['driving_style'],
                     margins=True, margins_name='Total')
    print("Tabla cruzada: Cluster descubierto vs Estilo auto-reportado")
    display(ct)

    # Heatmap de la tabla cruzada (sin totales)
    ct_clean = ct.drop('Total', axis=0).drop('Total', axis=1)
    ct_pct = ct_clean.div(ct_clean.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(ct_pct, annot=True, fmt='.1f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': '%'})
    ax.set_title('Cluster descubierto vs Estilo auto-reportado (%)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Estilo reportado en encuesta', fontsize=11)
    ax.set_ylabel('Cluster de conducción', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("Columna 'driving_style' no disponible. Saltando crosstab.")

In [ ]:
# Satisfacción por cluster de conducción (boxplot)
if df_merged is not None and 'satisfaction_score' in df_merged.columns:
    fig, ax = plt.subplots(figsize=(10, 6))

    cluster_palette = {i: plt.cm.Set2(i / max(optimal_k - 1, 1)) for i in range(optimal_k)}

    sns.boxplot(x='driving_cluster', y='satisfaction_score', data=df_merged,
                palette=cluster_palette, ax=ax, width=0.6)
    sns.stripplot(x='driving_cluster', y='satisfaction_score', data=df_merged,
                  color='black', alpha=0.3, size=3, ax=ax, jitter=True)

    ax.set_xlabel('Cluster de Conducción', fontsize=12)
    ax.set_ylabel('Satisfacción (1-5)', fontsize=12)
    ax.set_title('Satisfacción del cliente por patrón de conducción',
                 fontsize=14, fontweight='bold')

    # Medias
    means = df_merged.groupby('driving_cluster')['satisfaction_score'].mean()
    for i, mean_val in means.items():
        ax.text(i, mean_val + 0.05, f'\u03bc={mean_val:.2f}',
                ha='center', fontsize=10, fontweight='bold', color='#e74c3c')

    plt.tight_layout()
    plt.show()

    # Test estadístico
    from scipy import stats
    groups = [group['satisfaction_score'].dropna()
              for _, group in df_merged.groupby('driving_cluster')]
    if len(groups) >= 2 and all(len(g) > 0 for g in groups):
        stat, p_val = stats.kruskal(*groups)
        print(f"Kruskal-Wallis: H={stat:.3f}, p={p_val:.4f}")
        if p_val < 0.05:
            print("Diferencia significativa en satisfacción entre clusters (p < 0.05).")
        else:
            print("No hay diferencia significativa en satisfacción entre clusters.")
else:
    print("Columna 'satisfaction_score' no disponible. Saltando análisis.")

---
## 7. Resumen y Conclusiones

### Patrones descubiertos vs auto-reportados

El clustering basado en telemetría real revela patrones de conducción que pueden diferir
significativamente de lo que los conductores reportan sobre sí mismos. Esto tiene implicaciones
importantes:

- **Sesgo de auto-percepción**: Muchos conductores "agresivos" (según datos) se consideran
  "moderados" o "normales".
- **Oportunidad de intervención**: Programas de conducción eficiente pueden dirigirse a
  conductores cuyo patrón real difiere de su auto-percepción.

### Insights de negocio

1. **Seguros basados en uso (UBI)**: Los clusters de conducción permiten modelar riesgo real.
2. **Programas de eficiencia**: Identificar conductores con alto consumo que podrían mejorar.
3. **Gamificación**: Crear desafíos de conducción eficiente personalizados por patrón.
4. **Recomendación de vehículos**: Sugerir tipos de vehículo según estilo de conducción.

### Siguiente paso

> En el notebook [03 - Descomposición de Series Temporales](03_time_series_decomposition.ipynb)
> analizamos cómo evolucionan las métricas de conducción y eficiencia en el tiempo,
> buscando tendencias de degradación en vehículos específicos.